# Tech Challenge — Fase 1

**Tema:** Saúde da mulher — classificação de câncer de mama (maligno vs benigno)

**Objetivo:** atender às entregas técnicas do PDF — EDA, pré-processamento, dois ou mais modelos, métricas e explicabilidade.

## 1. Contexto e definição do problema

Redes de saúde feminina precisam **triagem rápida** para priorizar casos de risco. Este trabalho não substitui o médico: é um **apoio à decisão** baseado em padrões históricos dos dados.

- **Entrada:** medidas numéricas de núcleos celulares (ex.: raio, textura, perímetro).
- **Saída:** classe `malignant` (1) ou `benign` (0).
- **Métrica prioritária sugerida:** **recall** da classe maligna — errar um caso maligno (falso negativo) costuma ser mais grave que um falso positivo.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

## Dicionário de atributos

- **id**: Identificador único da amostra.
- **diagnosis**: Diagnóstico original no dataset (`M` = maligno, `B` = benigno).
- **radius_mean**: Raio médio dos núcleos celulares.
- **texture_mean**: Textura média dos núcleos celulares.
- **perimeter_mean**: Perímetro médio dos núcleos celulares.
- **area_mean**: Área média dos núcleos celulares.
- **smoothness_mean**: Suavidade média da borda dos núcleos.
- **compactness_mean**: Compacidade média dos núcleos.
- **concavity_mean**: Concavidade média das formas dos núcleos.
- **concave points_mean**: Número médio de pontos côncavos nos núcleos.
- **symmetry_mean**: Simetria média dos núcleos.
- **fractal_dimension_mean**: Dimensão fractal média dos núcleos.
- **radius_se**: Erro padrão do raio dos núcleos.
- **texture_se**: Erro padrão da textura dos núcleos.
- **perimeter_se**: Erro padrão do perímetro dos núcleos.
- **area_se**: Erro padrão da área dos núcleos.
- **smoothness_se**: Erro padrão da suavidade dos núcleos.
- **compactness_se**: Erro padrão da compacidade dos núcleos.
- **concavity_se**: Erro padrão da concavidade dos núcleos.
- **concave points_se**: Erro padrão do número de pontos côncavos.
- **symmetry_se**: Erro padrão da simetria dos núcleos.
- **fractal_dimension_se**: Erro padrão da dimensão fractal.
- **radius_worst**: Maior valor observado de raio nos núcleos.
- **texture_worst**: Maior valor observado de textura nos núcleos.
- **perimeter_worst**: Maior valor observado de perímetro nos núcleos.
- **area_worst**: Maior valor observado de área nos núcleos.
- **smoothness_worst**: Maior valor observado de suavidade nos núcleos.
- **compactness_worst**: Maior valor observado de compacidade nos núcleos.
- **concavity_worst**: Maior valor observado de concavidade nos núcleos.
- **concave points_worst**: Maior valor observado de pontos côncavos nos núcleos.
- **symmetry_worst**: Maior valor observado de simetria nos núcleos.
- **fractal_dimension_worst**: Maior dimensão fractal observada nos núcleos.

Observação: a coluna Unnamed: 32 pode ser descartada, pois não contém valores relevantes.

## 2. Carregamento e exploração dos dados

In [2]:
DATA_PATH = Path("..") / "data" / "data.csv"

df = pd.read_csv(DATA_PATH)

# Remover colunas vazias
df = df.dropna(axis=1, how='all')

# Alvo: M = maligno (1), B = benigno (0)
df["target"] = (df["diagnosis"].astype(str).str.upper() == "M").astype(int)

# Removendo colunas que não serão utilizadas
feature_cols = [c for c in df.columns if c not in ("id", "diagnosis", "target")]

print(f"Linhas: {df.shape[0]} | Colunas: {len(feature_cols)}")
print(df["diagnosis"].value_counts())
df.head()

Linhas: 569 | Colunas: 30
diagnosis
B    357
M    212
Name: count, dtype: int64


,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,target
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,1
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,1
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,1
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,1
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,1


#

In [33]:
df.info()
df.describe().T.head(10)

<class 'pandas.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 33 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       569 non-null    int64  
 1   diagnosis                569 non-null    str    
 2   radius_mean              569 non-null    float64
 3   texture_mean             569 non-null    float64
 4   perimeter_mean           569 non-null    float64
 5   area_mean                569 non-null    float64
 6   smoothness_mean          569 non-null    float64
 7   compactness_mean         569 non-null    float64
 8   concavity_mean           569 non-null    float64
 9   concave points_mean      569 non-null    float64
 10  symmetry_mean            569 non-null    float64
 11  fractal_dimension_mean   569 non-null    float64
 12  radius_se                569 non-null    float64
 13  texture_se               569 non-null    float64
 14  perimeter_se             569 non-null

,count,mean,std,min,25%,50%,75%,max
id,569.0,3.037183e+07,1.250206e+08,8670.00000,869218.00000,906024.00000,8.813129e+06,9.113205e+08
radius_mean,569.0,1.412729e+01,3.524049e+00,6.98100,11.70000,13.37000,1.578000e+01,2.811000e+01
texture_mean,569.0,1.928965e+01,4.301036e+00,9.71000,16.17000,18.84000,2.180000e+01,3.928000e+01
perimeter_mean,569.0,9.196903e+01,2.429898e+01,43.79000,75.17000,86.24000,1.041000e+02,1.885000e+02
area_mean,569.0,6.548891e+02,3.519141e+02,143.50000,420.30000,551.10000,7.827000e+02,2.501000e+03
smoothness_mean,569.0,9.636028e-02,1.406413e-02,0.05263,0.08637,0.09587,1.053000e-01,1.634000e-01
compactness_mean,569.0,1.043410e-01,5.281276e-02,0.01938,0.06492,0.09263,1.304000e-01,3.454000e-01
concavity_mean,569.0,8.879932e-02,7.971981e-02,0.00000,0.02956,0.06154,1.307000e-01,4.268000e-01
concave points_mean,569.0,4.891915e-02,3.880284e-02,0.00000,0.02031,0.03350,7.400000e-02,2.012000e-01
symmetry_mean,569.0,1.811619e-01,2.741428e-02,0.10600,0.16190,0.17920,1.957000e-01,3.040000e-01


## 3. Análise de correlação

## 4. Pré-processamento e separação treino/teste

## 5. Modelagem — três algoritmos

## 6. Explicabilidade — importância das variáveis e SHAP

## 7. Discussão crítica